In [1]:
# ==============================================================================
# 2026改修版: gcvEarth Strict (設計基準 A〜K 完全準拠)
# - 10-fold CVが全Fold成功したdegreeのみを採択する厳格版
# - 重要度: Permutation Importance (R2 drop) を適用
# ==============================================================================

suppressPackageStartupMessages({
  library(caret)
  library(dplyr)
  library(Metrics)
  library(earth)
})

set.seed(42)

# -------------------------------
# (F)(G) 設定：パスと対象ファイル
# -------------------------------
base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
file_names <- c(
  "n_base.csv", "n_base_OH_rebuilt.csv", "n_base_FP_rebuilt.csv",
  "n_all.csv",  "n_all_OH_rebuilt.csv",  "n_all_FP_rebuilt.csv",
  "m_base.csv", "m_base_OH_rebuilt.csv", "m_base_FP_rebuilt.csv",
  "m_all.csv",  "m_all_OH_rebuilt.csv",  "m_all_FP_rebuilt.csv"
)

# 出力先設定 (F)
today <- format(Sys.Date(), "%Y%m%d")
out_root <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/results"
model_name <- "gcvEarth"
run_dir <- file.path(out_root, paste0("Corr_1000_", model_name))
if (!dir.exists(run_dir)) dir.create(run_dir, recursive = TRUE)

# (C)(D)(E) 除外変数設定
inappropriate_vars <- c(
  'Jsccal', 'JscJsccal', 'PCEaTA', 'PCEaTAFinal', 'EQEABEST', 'Rshthin', 'Dresistance',
  'mobilityeblendOFET', 'mobilityep1MOFET', 'mobilityhblendSCLC', 'mobilityhnMSCLC', 'mobilityhp1MSCLC',
  'IonIoffF', 'DRMS', 'Var.1'
)
physical_exclude_patterns <- c("X2DGIWAXD", "X2DGIXD", "widthfibril")

# -------------------------------
# HELPERS
# -------------------------------
safe_r2 <- function(y, p) {
  r <- suppressWarnings(cor(y, p))
  if (is.na(r)) return(NA_real_)
  return(as.numeric(r^2))
}

# (J) Permutation Importance 関数
calc_earth_importance <- function(model, data_scaled, target, features) {
  base_pred <- predict(model, data_scaled[, features, drop = FALSE])
  base_r2 <- safe_r2(data_scaled[[target]], base_pred)
  
  out_imps <- list()
  for (col in features) {
    temp <- data_scaled; temp[[col]] <- sample(temp[[col]])
    shuffled_pred <- tryCatch(predict(model, temp[, features, drop = FALSE]), error = function(e) NA)
    new_r2 <- if (all(is.na(shuffled_pred))) 0 else safe_r2(data_scaled[[target]], shuffled_pred)
    out_imps[[col]] <- max(0, base_r2 - new_r2)
  }
  data.frame(Feature = names(out_imps), Importance = as.numeric(unlist(out_imps)))
}

# -------------------------------
# MAIN LOOP
# -------------------------------
summary_list <- list()
importance_list <- list()

for (file in file_names) {
  filepath <- file.path(base_path, file)
  if (!file.exists(filepath)) next
  
  df_raw <- tryCatch(read.csv(filepath, row.names = 1, check.names = FALSE), error = function(e) NULL)
  if (is.null(df_raw)) next
  if ("X" %in% colnames(df_raw)) df_raw$X <- NULL

  for (target in target_vars) {
    if (!target %in% colnames(df_raw)) next

    # クレンジング
    df_temp <- df_raw[, sapply(df_raw, is.numeric)] %>% na.omit()
    if (nrow(df_temp) < 20) next

    # (C)(D)(E) 変数除外
    features <- setdiff(colnames(df_temp), target_vars)
    features <- setdiff(features, inappropriate_vars)
    features <- features[!grepl(paste(physical_exclude_patterns, collapse="|"), features)]
    
    # (H) 多重共線性 & ゼロ分散除外
    vars <- sapply(df_temp[, features, drop = FALSE], var)
    features <- names(vars)[vars > 0 & !is.na(vars)]
    if (length(features) > 1) {
      cor_mat <- cor(df_temp[, features, drop = FALSE], use = "pairwise.complete.obs")
      high_corr <- findCorrelation(cor_mat, cutoff = 0.99999)
      if (length(high_corr) > 0) features <- features[-high_corr]
    }
    if (length(features) < 2) next

    # (I) スケーリング [-1, 1]
    pp <- preProcess(df_temp[, features, drop = FALSE], method = "range")
    df_scaled <- df_temp
    df_scaled[, features] <- predict(pp, df_temp[, features]) * 2 - 1

    # (K) CV10設定 (A)OOD不要
    train_ctrl <- trainControl(method = "cv", number = 10, savePredictions = "final")

    # --- 厳格なDegree選択 ---
    set.seed(42)
    best_model <- NULL; models_ok <- list(); rmse_ok <- c()
    
    for (deg in 1:3) {
      m_tmp <- tryCatch(
        train(x = df_scaled[, features], y = df_scaled[[target]], method = "gcvEarth", 
              trControl = train_ctrl, tuneGrid = expand.grid(degree = deg),
              nk = 80, minspan = 25, penalty = 4, nprune = 40),
        error = function(e) NULL
      )
      # 全Fold成功チェック
      if (!is.null(m_tmp) && !is.null(m_tmp$resample) && nrow(m_tmp$resample) == 10) {
        if (!any(is.na(m_tmp$resample$RMSE))) {
          models_ok[[as.character(deg)]] <- m_tmp
          rmse_ok[as.character(deg)] <- m_tmp$results$RMSE[1]
        }
      }
    }
    
    if (length(models_ok) > 0) {
      best_model <- models_ok[[names(rmse_ok)[which.min(rmse_ok)]]]
    } else { next }

    # --- 保存処理 ---
    tag <- paste0(tools::file_path_sans_ext(file), "_", target)
    saveRDS(best_model, file.path(run_dir, paste0(tag, "_model.rds")))

    # (B) CV10_OOF予測CSV
    pred_oof <- best_model$pred %>% 
      filter(degree == best_model$bestTune$degree) %>%
      mutate(SampleID = rownames(df_scaled)[rowIndex], Type = "CV10_OOF") %>%
      select(SampleID, Observed = obs, Predicted = pred, Type)
    write.csv(pred_oof, file.path(run_dir, paste0(tag, "_pred_OOF.csv")), row.names = FALSE)

    # (J) 重要度
    imp_df <- calc_earth_importance(best_model, df_scaled, target, features)
    imp_df$File <- file; imp_df$Target <- target
    importance_list[[length(importance_list)+1]] <- imp_df

    # (K) サマリー
    summary_list[[length(summary_list)+1]] <- data.frame(
      File=file, Target=target, 
      CV10_R2=safe_r2(pred_oof$Observed, pred_oof$Predicted),
      CV10_RMSE=rmse(pred_oof$Observed, pred_oof$Predicted), 
      n_samples=nrow(df_scaled), n_features=length(features), 
      Best_Degree=best_model$bestTune$degree
    )
    
    cat(sprintf("  Finished: %s - %s | Degree=%d | CV10_R2: %.3f\n", 
                file, target, best_model$bestTune$degree, tail(summary_list, 1)[[1]]$CV10_R2))
  }
}

# CSV出力
if (length(summary_list) > 0) {
  write.csv(do.call(rbind, summary_list), file.path(run_dir, "all_summary_CV10.csv"), row.names = FALSE)
  write.csv(do.call(rbind, importance_list), file.path(run_dir, "all_importance_gcvEarth.csv"), row.names = FALSE)
}

cat("\n[DONE] gcvEarth Analysis completed.\n")

  Finished: n_base.csv - Jsc | Degree=2 | CV10_R2: 0.520
  Finished: n_base.csv - Voc | Degree=1 | CV10_R2: 0.796
  Finished: n_base.csv - FF | Degree=3 | CV10_R2: 0.164
  Finished: n_base.csv - PCEmax | Degree=1 | CV10_R2: 0.566
  Finished: n_base_OH_rebuilt.csv - Jsc | Degree=2 | CV10_R2: 0.732
  Finished: n_base_OH_rebuilt.csv - Voc | Degree=1 | CV10_R2: 0.804


Warning message:
"model fit failed for Fold05: degree=2 Error in leaps.setup(x = bx, y = y, force.in = 1, force.out = NULL, intercept = FALSE,  : 
  NA/NaN/Inf in foreign function call (arg 3)
"
Warning message in nominalTrainWorkflow(x = x, y = y, wts = weights, info = trainInfo, :
"There were missing values in resampled performance measures."


  Finished: n_base_OH_rebuilt.csv - FF | Degree=1 | CV10_R2: 0.371
  Finished: n_base_OH_rebuilt.csv - PCEmax | Degree=1 | CV10_R2: 0.673
  Finished: n_base_FP_rebuilt.csv - Jsc | Degree=3 | CV10_R2: 0.678
  Finished: n_base_FP_rebuilt.csv - Voc | Degree=1 | CV10_R2: 0.773
  Finished: n_base_FP_rebuilt.csv - FF | Degree=1 | CV10_R2: 0.387
  Finished: n_base_FP_rebuilt.csv - PCEmax | Degree=1 | CV10_R2: 0.628
  Finished: n_all.csv - Jsc | Degree=1 | CV10_R2: 0.569
  Finished: n_all.csv - Voc | Degree=1 | CV10_R2: 0.604
  Finished: n_all.csv - FF | Degree=3 | CV10_R2: 0.377
  Finished: n_all.csv - PCEmax | Degree=1 | CV10_R2: 0.490
  Finished: n_all_OH_rebuilt.csv - Jsc | Degree=2 | CV10_R2: 0.604
  Finished: n_all_OH_rebuilt.csv - Voc | Degree=1 | CV10_R2: 0.533
  Finished: n_all_OH_rebuilt.csv - FF | Degree=1 | CV10_R2: 0.389
  Finished: n_all_OH_rebuilt.csv - PCEmax | Degree=1 | CV10_R2: 0.500
  Finished: n_all_FP_rebuilt.csv - Jsc | Degree=2 | CV10_R2: 0.612
  Finished: n_all_FP_rebu

Warning message:
"model fit failed for Fold02: degree=2 Error in leaps.setup(x = bx, y = y, force.in = 1, force.out = NULL, intercept = FALSE,  : 
  NA/NaN/Inf in foreign function call (arg 3)
"
Warning message:
"model fit failed for Fold08: degree=2 Error in leaps.setup(x = bx, y = y, force.in = 1, force.out = NULL, intercept = FALSE,  : 
  NA/NaN/Inf in foreign function call (arg 3)
"
Warning message in nominalTrainWorkflow(x = x, y = y, wts = weights, info = trainInfo, :
"There were missing values in resampled performance measures."
Warning message:
"model fit failed for Fold01: degree=3 Error in leaps.setup(x = bx, y = y, force.in = 1, force.out = NULL, intercept = FALSE,  : 
  NA/NaN/Inf in foreign function call (arg 3)
"
Warning message:
"model fit failed for Fold05: degree=3 Error in leaps.setup(x = bx, y = y, force.in = 1, force.out = NULL, intercept = FALSE,  : 
  NA/NaN/Inf in foreign function call (arg 3)
"
Warning message:
"model fit failed for Fold06: degree=3 Error in lea

  Finished: m_base_OH_rebuilt.csv - FF | Degree=1 | CV10_R2: 0.451


Warning message:
"model fit failed for Fold02: degree=2 Error in leaps.setup(x = bx, y = y, force.in = 1, force.out = NULL, intercept = FALSE,  : 
  NA/NaN/Inf in foreign function call (arg 3)
"
Warning message in nominalTrainWorkflow(x = x, y = y, wts = weights, info = trainInfo, :
"There were missing values in resampled performance measures."
Warning message:
"model fit failed for Fold02: degree=3 Error in leaps.setup(x = bx, y = y, force.in = 1, force.out = NULL, intercept = FALSE,  : 
  NA/NaN/Inf in foreign function call (arg 3)
"
Warning message:
"model fit failed for Fold10: degree=3 Error in leaps.setup(x = bx, y = y, force.in = 1, force.out = NULL, intercept = FALSE,  : 
  NA/NaN/Inf in foreign function call (arg 3)
"
Warning message in nominalTrainWorkflow(x = x, y = y, wts = weights, info = trainInfo, :
"There were missing values in resampled performance measures."


  Finished: m_base_OH_rebuilt.csv - PCEmax | Degree=1 | CV10_R2: 0.563
  Finished: m_base_FP_rebuilt.csv - Jsc | Degree=1 | CV10_R2: 0.552
  Finished: m_base_FP_rebuilt.csv - Voc | Degree=1 | CV10_R2: 0.597
  Finished: m_base_FP_rebuilt.csv - FF | Degree=3 | CV10_R2: 0.383
  Finished: m_base_FP_rebuilt.csv - PCEmax | Degree=1 | CV10_R2: 0.590
  Finished: m_all.csv - Jsc | Degree=1 | CV10_R2: 0.488
  Finished: m_all.csv - Voc | Degree=2 | CV10_R2: 0.462
  Finished: m_all.csv - FF | Degree=2 | CV10_R2: 0.392
  Finished: m_all.csv - PCEmax | Degree=1 | CV10_R2: 0.520
  Finished: m_all_OH_rebuilt.csv - Jsc | Degree=1 | CV10_R2: 0.608
  Finished: m_all_OH_rebuilt.csv - Voc | Degree=2 | CV10_R2: 0.535
  Finished: m_all_OH_rebuilt.csv - FF | Degree=1 | CV10_R2: 0.459
  Finished: m_all_OH_rebuilt.csv - PCEmax | Degree=1 | CV10_R2: 0.598
  Finished: m_all_FP_rebuilt.csv - Jsc | Degree=1 | CV10_R2: 0.598
  Finished: m_all_FP_rebuilt.csv - Voc | Degree=1 | CV10_R2: 0.560
  Finished: m_all_FP_rebu